# 论文可信引用包 Evidence Pack

> 将 claim / quote / doc_id / chunk_id / offset / page_no / title 标准化，RAG/Agent 的底层模板

**场景**: 所有 RAG/Agent 案例都需要一个标准化的引用包格式。本案例定义了 Evidence Pack 的标准结构，并展示如何从 Sciverse 检索结果构建它。

**使用接口**: agentic-search, content

**预估调用量**: ~8–15 次 API 调用 / 一次构建

---


## Step 1: 环境准备

安装依赖并配置 API Token


In [ ]:
!pip install httpx
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值


## Step 2: 定义 Evidence Pack 标准结构

定义标准化的引用包数据结构


In [ ]:
from dataclasses import dataclass, asdict
from typing import Optional
import json

@dataclass
class EvidenceItem:
    claim: str
    quote: str
    doc_id: str
    offset: int
    title: str
    venue: Optional[str] = None
    year: Optional[int] = None
    confidence: float = 0.0

@dataclass
class EvidencePack:
    items: list[EvidenceItem]

    def to_json(self) -> str:
        return json.dumps({"evidence_pack": [asdict(i) for i in self.items]}, ensure_ascii=False, indent=2)

# \u793a\u4f8b
pack = EvidencePack(items=[])
print(pack.to_json())


## Step 3: 为每个 claim 检索并构建引用

逐条检索 claim 对应的文献证据


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def build_evidence(claim: str) -> Optional[EvidenceItem]:
    """\u4e3a\u5355\u4e2a claim \u68c0\u7d22\u5e76\u6784\u5efa\u5f15\u7528"""
    async with httpx.AsyncClient(timeout=30) as client:
        # Step 1: \u8bed\u4e49\u68c0\u7d22
        resp = await client.post(
            f"{BASE}/agentic-search", headers=HEADERS,
            json={"query": claim, "top_k": 5}
        )
        resp.raise_for_status()
        hits = resp.json()["hits"]
        if not hits:
            return None
        best = hits[0]

        # Step 2: \u8bfb\u53d6\u539f\u6587\u5b9a\u4f4d\u786e\u5207\u5f15\u7528
        resp2 = await client.get(
            f"{BASE}/content", headers=HEADERS,
            params={"doc_id": best["doc_id"], "offset": best.get("offset", 0), "limit": 800}
        )
        resp2.raise_for_status()
        text = resp2.json()["text"]

        return EvidenceItem(
            claim=claim,
            quote=text[:200],  # \u53d6\u524d 200 \u5b57\u7b26\u4f5c\u4e3a\u5f15\u7528
            doc_id=best["doc_id"],
            offset=best.get("offset", 0),
            title=best["title"],
            confidence=best["score"]
        )

claims = [
    "AlphaFold2 \u5728 CASP14 \u4e2d\u8fbe\u5230\u4e86\u5b9e\u9a8c\u7cbe\u5ea6",
    "mRNA \u7684 LNP \u9012\u9001\u7cfb\u7edf\u663e\u8457\u63d0\u9ad8\u4e86\u7ec6\u80de\u5185\u5316\u6548\u7387",
]

async def main():
    items = []
    for claim in claims:
        evidence = await build_evidence(claim)
        if evidence:
            items.append(evidence)
            print(f"\u2713 {claim[:40]}... -> {evidence.doc_id}")
        else:
            print(f"\u2717 {claim[:40]}... -> no evidence found")
    pack = EvidencePack(items=items)
    print(f"\
Evidence Pack ({len(items)}/{len(claims)} claims grounded):")
    print(pack.to_json())

asyncio.run(main())


## 注意事项

- Evidence Pack 是所有 RAG/Agent 案例的底层模板，建议封装为通用工具
- confidence 基于 agentic-search 的 score，可结合来源权威性进一步调整
- quote 应从 content 返回的 text 中截取，而非 LLM 生成
- 可扩展字段：page_no、chunk_id、section_title 等
- 生产环境建议对每个 claim 并发检索以提高速度


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
